# Computação Gráfica - Projeto 1
## Cena: Astronauta no Espaço

### Felipe da Costa Coqueiro, NUSP: 11781361
### Fernando Alee Suaiden, NUSP: 12680836

### Controles:
- **W/A/S/D**: Mover astronauta (translação)
- **Z/X**: Diminuir/Aumentar foguete (escala)
- **Q/E**: Rotacionar estrela/sol
- **P**: Mostrar/ocultar malha poligonal (wireframe)
- **ESC**: Sair

### Objetos:
1. **Astronauta** (3D) - Composto por esferas e cilindros
2. **Foguete** (3D) - Composto por cone e cilindros
3. **Estrela/Sol** (2D) - Composto por triângulos
4. **Planeta** (3D) - Esfera com anéis (tipo Saturno)
5. **Satélite** (2D) - Composto por retângulos e painéis solares

## 1. Instalação das Dependências

In [1]:
# Instalar bibliotecas necessárias
!pip install PyOpenGL glfw numpy

  Using cached pyopengl-3.1.10-py3-none-any.whl.metadata (3.3 kB)
  Using cached glfw-2.10.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38.py39.py310.py311.py312.py313.py314-none-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached pyopengl-3.1.10-py3-none-any.whl (3.2 MB)
Using cached glfw-2.10.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38.py39.py310.py311.py312.py313.py314-none-manylinux_2_28_x86_64.whl (243 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [numpy]32m2/3 [numpy]


## 2. Importações e Configurações Globais

In [ ]:
import glfw
from OpenGL.GL import *
import numpy as np
import math
import ctypes

# Dimensões da janela
WINDOW_WIDTH = 1200
WINDOW_HEIGHT = 800

# Estado do programa
wireframe_mode = False

# Transformações dos objetos
astronauta_pos = [0.0, 0.0, 0.0]      # Posição do astronauta (translação)
foguete_scale = 1.0                   # Escala do foguete
estrela_rotation = 0.0                # Rotação da estrela
planeta_rotation = 0.0                # Rotação automática do planeta

# Velocidades
TRANSLATION_SPEED = 0.05
SCALE_SPEED = 0.05
ROTATION_SPEED = 3.0

print("Configurações carregadas com sucesso!")

Configurações carregadas com sucesso!


## 3. Shaders GLSL

Utilizamos shaders modernos do OpenGL (versão 3.3 core profile):
- **Vertex Shader**: Recebe vértices já transformados pela CPU (sem matrizes uniform)
- **Fragment Shader**: Define a cor final de cada fragmento

Todas as transformações geométricas (translação, escala, rotação e projeção ortográfica) são aplicadas diretamente nos vértices via CPU antes do envio ao GPU.

In [ ]:
# ============================================================================
# SHADERS GLSL
# ============================================================================

# Sem uniforms de matrizes (projection, model, view).
# Todas as transformações são aplicadas nos vértices via CPU.

vertex_shader_source = """
#version 330 core

layout(location = 0) in vec3 aPos;
layout(location = 1) in vec3 aColor;

out vec3 vertexColor;

void main() {
    gl_Position = vec4(aPos, 1.0);
    vertexColor = aColor;
}
"""

fragment_shader_source = """
#version 330 core

in vec3 vertexColor;
out vec4 FragColor;

void main() {
    FragColor = vec4(vertexColor, 1.0);
}
"""

print("Shaders definidos!")

Shaders definidos!


## 4. Funções de Compilação de Shaders

In [4]:
# ============================================================================
# FUNÇÕES AUXILIARES PARA SHADERS
# ============================================================================

def compile_shader(source, shader_type):
    """Compila um shader GLSL."""
    shader = glCreateShader(shader_type)
    glShaderSource(shader, source)
    glCompileShader(shader)
    
    # Verifica erros de compilação
    if not glGetShaderiv(shader, GL_COMPILE_STATUS):
        error = glGetShaderInfoLog(shader).decode()
        raise RuntimeError(f"Erro ao compilar shader: {error}")
    
    return shader

def create_shader_program():
    """Cria o programa de shaders."""
    vertex_shader = compile_shader(vertex_shader_source, GL_VERTEX_SHADER)
    fragment_shader = compile_shader(fragment_shader_source, GL_FRAGMENT_SHADER)
    
    program = glCreateProgram()
    glAttachShader(program, vertex_shader)
    glAttachShader(program, fragment_shader)
    glLinkProgram(program)
    
    # Verifica erros de linkagem
    if not glGetProgramiv(program, GL_LINK_STATUS):
        error = glGetProgramInfoLog(program).decode()
        raise RuntimeError(f"Erro ao linkar programa: {error}")
    
    # Limpa shaders (já linkados)
    glDeleteShader(vertex_shader)
    glDeleteShader(fragment_shader)
    
    return program

print("Funções de shader definidas!")

Funções de shader definidas!


## 5. Matrizes de Transformação

Implementamos manualmente as matrizes de transformação geométrica:
- **Translação**: Move objetos no espaço
- **Escala**: Redimensiona objetos
- **Rotação**: Gira objetos em torno dos eixos X, Y e Z
- **Projeção Ortográfica**: Define a visualização da cena

In [5]:
# ============================================================================
# FUNÇÕES DE MATRIZES DE TRANSFORMAÇÃO
# ============================================================================

def matrix_identity():
    """Retorna matriz identidade 4x4."""
    return np.identity(4, dtype=np.float32)

def matrix_translation(tx, ty, tz):
    """Retorna matriz de translação."""
    mat = np.identity(4, dtype=np.float32)
    mat[0, 3] = tx
    mat[1, 3] = ty
    mat[2, 3] = tz
    return mat

def matrix_scale(sx, sy, sz):
    """Retorna matriz de escala."""
    mat = np.identity(4, dtype=np.float32)
    mat[0, 0] = sx
    mat[1, 1] = sy
    mat[2, 2] = sz
    return mat

def matrix_rotation_x(angle_degrees):
    """Retorna matriz de rotação em torno do eixo X."""
    angle = math.radians(angle_degrees)
    c, s = math.cos(angle), math.sin(angle)
    mat = np.identity(4, dtype=np.float32)
    mat[1, 1] = c
    mat[1, 2] = -s
    mat[2, 1] = s
    mat[2, 2] = c
    return mat

def matrix_rotation_y(angle_degrees):
    """Retorna matriz de rotação em torno do eixo Y."""
    angle = math.radians(angle_degrees)
    c, s = math.cos(angle), math.sin(angle)
    mat = np.identity(4, dtype=np.float32)
    mat[0, 0] = c
    mat[0, 2] = s
    mat[2, 0] = -s
    mat[2, 2] = c
    return mat

def matrix_rotation_z(angle_degrees):
    """Retorna matriz de rotação em torno do eixo Z."""
    angle = math.radians(angle_degrees)
    c, s = math.cos(angle), math.sin(angle)
    mat = np.identity(4, dtype=np.float32)
    mat[0, 0] = c
    mat[0, 1] = -s
    mat[1, 0] = s
    mat[1, 1] = c
    return mat

def matrix_orthographic(left, right, bottom, top, near, far):
    """Retorna matriz de projeção ortográfica."""
    mat = np.zeros((4, 4), dtype=np.float32)
    mat[0, 0] = 2.0 / (right - left)
    mat[1, 1] = 2.0 / (top - bottom)
    mat[2, 2] = -2.0 / (far - near)
    mat[0, 3] = -(right + left) / (right - left)
    mat[1, 3] = -(top + bottom) / (top - bottom)
    mat[2, 3] = -(far + near) / (far - near)
    mat[3, 3] = 1.0
    return mat

def matrix_multiply(a, b):
    """Multiplica duas matrizes 4x4."""
    return np.dot(a, b)

print("Matrizes de transformação definidas!")

Matrizes de transformação definidas!


## 6. Funções de Criação de Geometrias

Criamos primitivas geométricas 3D e 2D a partir do zero:
- **Esfera**: Para capacete, planeta, luvas
- **Cilindro**: Para corpo, braços, pernas
- **Cone**: Para ponta do foguete, aletas
- **Torus**: Para anéis do planeta
- **Estrela 2D**: Para o sol
- **Retângulo**: Para painéis solares do satélite

In [6]:
# ============================================================================
# FUNÇÕES PARA CRIAR GEOMETRIAS
# ============================================================================

def create_sphere(radius, slices, stacks, color):
    """
    Cria uma esfera 3D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    vertices = []
    indices = []
    
    for i in range(stacks + 1):
        phi = math.pi * i / stacks
        for j in range(slices + 1):
            theta = 2.0 * math.pi * j / slices
            
            x = radius * math.sin(phi) * math.cos(theta)
            y = radius * math.cos(phi)
            z = radius * math.sin(phi) * math.sin(theta)
            
            vertices.extend([x, y, z, color[0], color[1], color[2]])
    
    for i in range(stacks):
        for j in range(slices):
            first = i * (slices + 1) + j
            second = first + slices + 1
            
            indices.extend([first, second, first + 1])
            indices.extend([second, second + 1, first + 1])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)

def create_cylinder(radius, height, slices, color):
    """
    Cria um cilindro 3D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    vertices = []
    indices = []
    
    # Vértices do corpo
    for i in range(2):
        y = -height/2 + i * height
        for j in range(slices + 1):
            theta = 2.0 * math.pi * j / slices
            x = radius * math.cos(theta)
            z = radius * math.sin(theta)
            vertices.extend([x, y, z, color[0], color[1], color[2]])
    
    # Centro inferior e superior
    center_bottom_idx = len(vertices) // 6
    vertices.extend([0, -height/2, 0, color[0], color[1], color[2]])
    center_top_idx = len(vertices) // 6
    vertices.extend([0, height/2, 0, color[0], color[1], color[2]])
    
    # Índices do corpo
    for j in range(slices):
        first = j
        second = j + slices + 1
        indices.extend([first, second, first + 1])
        indices.extend([second, second + 1, first + 1])
    
    # Índices da base inferior
    for j in range(slices):
        indices.extend([center_bottom_idx, j + 1, j])
    
    # Índices da base superior
    for j in range(slices):
        indices.extend([center_top_idx, slices + 1 + j, slices + 1 + j + 1])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)

def create_cone(radius, height, slices, color):
    """
    Cria um cone 3D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    vertices = []
    indices = []
    
    # Vértice do topo
    vertices.extend([0, height/2, 0, color[0], color[1], color[2]])
    
    # Vértices da base
    for j in range(slices + 1):
        theta = 2.0 * math.pi * j / slices
        x = radius * math.cos(theta)
        z = radius * math.sin(theta)
        vertices.extend([x, -height/2, z, color[0], color[1], color[2]])
    
    # Centro da base
    center_idx = len(vertices) // 6
    vertices.extend([0, -height/2, 0, color[0], color[1], color[2]])
    
    # Índices do corpo cônico
    for j in range(slices):
        indices.extend([0, j + 1, j + 2])
    
    # Índices da base
    for j in range(slices):
        indices.extend([center_idx, j + 2, j + 1])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)

def create_torus(major_radius, minor_radius, major_segments, minor_segments, color):
    """
    Cria um torus (anel) 3D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    vertices = []
    indices = []
    
    for i in range(major_segments + 1):
        theta = 2.0 * math.pi * i / major_segments
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
        
        for j in range(minor_segments + 1):
            phi = 2.0 * math.pi * j / minor_segments
            cos_phi = math.cos(phi)
            sin_phi = math.sin(phi)
            
            x = (major_radius + minor_radius * cos_phi) * cos_theta
            y = minor_radius * sin_phi
            z = (major_radius + minor_radius * cos_phi) * sin_theta
            
            vertices.extend([x, y, z, color[0], color[1], color[2]])
    
    for i in range(major_segments):
        for j in range(minor_segments):
            first = i * (minor_segments + 1) + j
            second = first + minor_segments + 1
            
            indices.extend([first, second, first + 1])
            indices.extend([second, second + 1, first + 1])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)

def create_star_2d(outer_radius, inner_radius, points, color):
    """
    Cria uma estrela 2D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    vertices = []
    indices = []
    
    # Centro
    vertices.extend([0, 0, 0, color[0], color[1], color[2]])
    
    # Pontas da estrela
    for i in range(points * 2):
        angle = math.pi / 2 + i * math.pi / points
        radius = outer_radius if i % 2 == 0 else inner_radius
        x = radius * math.cos(angle)
        y = radius * math.sin(angle)
        vertices.extend([x, y, 0, color[0], color[1], color[2]])
    
    # Triângulos
    for i in range(points * 2):
        next_i = (i + 1) % (points * 2)
        indices.extend([0, i + 1, next_i + 1])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)

def create_rectangle(width, height, color):
    """
    Cria um retângulo 2D.
    Retorna vértices e índices para GL_TRIANGLES.
    """
    hw, hh = width / 2, height / 2
    vertices = np.array([
        -hw, -hh, 0, color[0], color[1], color[2],
         hw, -hh, 0, color[0], color[1], color[2],
         hw,  hh, 0, color[0], color[1], color[2],
        -hw,  hh, 0, color[0], color[1], color[2],
    ], dtype=np.float32)
    indices = np.array([0, 1, 2, 0, 2, 3], dtype=np.uint32)
    return vertices, indices

print("Funções de geometria definidas!")

Funções de geometria definidas!


## 7. Classe para Gerenciar Objetos OpenGL

In [ ]:
# ============================================================================
# CLASSE PARA GERENCIAR OBJETOS
# ============================================================================

class Objeto3D:
    """Classe para gerenciar um objeto renderizável."""
    
    def __init__(self, vertices, indices):
        """Inicializa o objeto com vértices e índices."""
        self.original_vertices = vertices.copy()
        self.num_vertices = len(vertices) // 6
        self.num_indices = len(indices)
        
        # Cria VAO
        self.vao = glGenVertexArrays(1)
        glBindVertexArray(self.vao)
        
        # Cria VBO (DYNAMIC pois atualizamos cada frame)
        self.vbo = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.vbo)
        glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
        
        # Cria EBO
        self.ebo = glGenBuffers(1)
        glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, self.ebo)
        glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices.nbytes, indices, GL_STATIC_DRAW)
        
        # Configura atributos de vértice
        stride = 6 * vertices.itemsize
        
        # Posição (location = 0)
        glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))
        glEnableVertexAttribArray(0)
        
        # Cor (location = 1)
        glVertexAttribPointer(1, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(3 * vertices.itemsize))
        glEnableVertexAttribArray(1)
        
        glBindVertexArray(0)
    
    def draw(self, transform=None):
        """Desenha o objeto, aplicando a transformação nos vértices via CPU."""
        if transform is not None:
            transformed = self.original_vertices.copy()
            reshaped = transformed.reshape(-1, 6)
            positions = reshaped[:, :3]
            ones = np.ones((positions.shape[0], 1), dtype=np.float32)
            pos4 = np.hstack([positions, ones])
            new_pos4 = (transform @ pos4.T).T
            reshaped[:, :3] = new_pos4[:, :3]
            glBindBuffer(GL_ARRAY_BUFFER, self.vbo)
            glBufferSubData(GL_ARRAY_BUFFER, 0, transformed.nbytes, transformed)
        
        glBindVertexArray(self.vao)
        glDrawElements(GL_TRIANGLES, self.num_indices, GL_UNSIGNED_INT, None)
        glBindVertexArray(0)
    
    def cleanup(self):
        """Libera recursos."""
        glDeleteVertexArrays(1, [self.vao])
        glDeleteBuffers(1, [self.vbo])
        glDeleteBuffers(1, [self.ebo])

print("Classe Objeto3D definida!")

Classe Objeto3D definida!


## 8. Criação dos Objetos da Cena

### 8.1 Astronauta (3D)
Composto por: capacete (esfera), visor (esfera), corpo (cilindro), mochila (cilindro), braços (cilindros), luvas (esferas), pernas (cilindros), botas (esferas)

In [8]:
def create_astronauta():
    """
    Cria o astronauta composto por várias partes.
    Retorna lista de objetos e suas transformações locais.
    """
    partes = []
    
    # Cores do astronauta (branco/cinza para o traje)
    cor_capacete = [0.9, 0.9, 0.95]     # Branco azulado
    cor_traje = [0.95, 0.95, 0.95]      # Branco
    cor_visor = [0.2, 0.3, 0.5]         # Azul escuro (visor)
    cor_detalhes = [0.8, 0.4, 0.1]      # Laranja (detalhes)
    cor_mochila = [0.6, 0.6, 0.65]      # Cinza (mochila)
    
    # Capacete (esfera)
    v, i = create_sphere(0.12, 16, 16, cor_capacete)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.28, 0)))
    
    # Visor do capacete (esfera menor na frente)
    v, i = create_sphere(0.08, 12, 12, cor_visor)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.28, 0.06)))
    
    # Corpo (cilindro)
    v, i = create_cylinder(0.1, 0.25, 16, cor_traje)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.08, 0)))
    
    # Mochila de oxigênio (cubo aproximado por cilindro)
    v, i = create_cylinder(0.07, 0.2, 8, cor_mochila)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.1, -0.1)))
    
    # Braço esquerdo (cilindro)
    v, i = create_cylinder(0.035, 0.18, 12, cor_traje)
    braco_esquerdo_rot = matrix_rotation_z(150)
    braco_esquerdo_trans = matrix_translation(-0.15, 0.1, 0)
    braco_esquerdo_transform = matrix_multiply(braco_esquerdo_trans, braco_esquerdo_rot)
    partes.append((Objeto3D(v, i), braco_esquerdo_transform))
    
    # Braço direito (cilindro)
    v, i = create_cylinder(0.035, 0.18, 12, cor_traje)
    braco_direito_rot = matrix_rotation_z(-150)
    braco_direito_trans = matrix_translation(0.15, 0.1, 0)
    braco_direito_transform = matrix_multiply(braco_direito_trans, braco_direito_rot)
    partes.append((Objeto3D(v, i), braco_direito_transform))
    
    # Luva esquerda (alinhada ao fim do braço)
    v, i = create_sphere(0.04, 10, 10, cor_detalhes)
    luva_esquerda_offset = matrix_translation(0, 0.11, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(braco_esquerdo_transform, luva_esquerda_offset)))
    
    # Luva direita (alinhada ao fim do braço)
    v, i = create_sphere(0.04, 10, 10, cor_detalhes)
    luva_direita_offset = matrix_translation(0, 0.11, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(braco_direito_transform, luva_direita_offset)))
    
    # Perna esquerda (cilindro)
    v, i = create_cylinder(0.04, 0.2, 12, cor_traje)
    partes.append((Objeto3D(v, i), matrix_translation(-0.05, -0.15, 0)))
    
    # Perna direita (cilindro)
    v, i = create_cylinder(0.04, 0.2, 12, cor_traje)
    partes.append((Objeto3D(v, i), matrix_translation(0.05, -0.15, 0)))
    
    # Bota esquerda (esfera achatada)
    v, i = create_sphere(0.045, 10, 10, cor_detalhes)
    partes.append((Objeto3D(v, i), matrix_translation(-0.05, -0.28, 0.02)))
    
    # Bota direita (esfera achatada)
    v, i = create_sphere(0.045, 10, 10, cor_detalhes)
    partes.append((Objeto3D(v, i), matrix_translation(0.05, -0.28, 0.02)))
    
    return partes

print("Função create_astronauta definida!")

Função create_astronauta definida!


### 8.2 Foguete (3D)
Composto por: ponta (cone), corpo (cilindro), janelas (esferas), propulsor (cilindro), chama (cone), aletas (cones)

In [9]:
def create_foguete():
    """
    Cria um foguete composto por cone e cilindros.
    Retorna lista de objetos e suas transformações locais.
    """
    partes = []
    
    # Cores do foguete
    cor_corpo = [0.85, 0.85, 0.9]       # Branco/prata
    cor_ponta = [0.9, 0.2, 0.1]         # Vermelho
    cor_janela = [0.3, 0.5, 0.8]        # Azul
    cor_propulsor = [0.3, 0.3, 0.35]    # Cinza escuro
    cor_chama = [1.0, 0.6, 0.1]         # Laranja (chama)
    
    # Ponta do foguete (cone)
    v, i = create_cone(0.08, 0.15, 20, cor_ponta)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.35, 0)))
    
    # Corpo principal (cilindro)
    v, i = create_cylinder(0.08, 0.4, 20, cor_corpo)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.08, 0)))
    
    # Janela (esfera pequena na frente)
    v, i = create_sphere(0.03, 12, 12, cor_janela)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.15, 0.075)))
    
    # Segunda janela
    v, i = create_sphere(0.025, 10, 10, cor_janela)
    partes.append((Objeto3D(v, i), matrix_translation(0, 0.05, 0.078)))
    
    # Propulsor principal (cilindro)
    v, i = create_cylinder(0.06, 0.1, 16, cor_propulsor)
    partes.append((Objeto3D(v, i), matrix_translation(0, -0.17, 0)))
    
    # Chama do propulsor (cone invertido)
    v, i = create_cone(0.05, 0.12, 16, cor_chama)
    rot = matrix_rotation_x(180)
    trans = matrix_translation(0, -0.28, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(trans, rot)))
    
    # Aleta 1 (cone lateral esquerdo)
    v, i = create_cone(0.03, 0.12, 12, cor_ponta)
    rot = matrix_rotation_z(130)
    trans = matrix_translation(-0.11, -0.16, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(trans, rot)))
    
    # Aleta 2 (cone lateral direito)
    v, i = create_cone(0.03, 0.12, 12, cor_ponta)
    rot = matrix_rotation_z(-130)
    trans = matrix_translation(0.11, -0.16, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(trans, rot)))
    
    # Aleta 3 (cone traseiro)
    v, i = create_cone(0.03, 0.12, 12, cor_ponta)
    rot = matrix_rotation_x(-130)
    trans = matrix_translation(0, -0.16, -0.11)
    partes.append((Objeto3D(v, i), matrix_multiply(trans, rot)))
    
    return partes

print("Função create_foguete definida!")

Função create_foguete definida!


### 8.3 Estrela/Sol (2D)
Composto por: centro (esfera), raios externos (estrela 2D), raios internos (estrela 2D rotacionada)

In [10]:
def create_estrela_sol():
    """
    Cria uma estrela/sol 2D composta por triângulos.
    Retorna lista de objetos e suas transformações locais.
    """
    partes = []
    
    # Cores
    cor_centro = [1.0, 0.9, 0.3]        # Amarelo brilhante
    cor_raios = [1.0, 0.7, 0.1]         # Laranja/amarelo
    
    # Centro da estrela (esfera pequena para dar volume)
    v, i = create_sphere(0.08, 16, 16, cor_centro)
    partes.append((Objeto3D(v, i), matrix_identity()))
    
    # Raios da estrela (estrela 2D)
    v, i = create_star_2d(0.2, 0.1, 8, cor_raios)
    partes.append((Objeto3D(v, i), matrix_identity()))
    
    # Segunda camada de raios (menores, rotacionados)
    v, i = create_star_2d(0.15, 0.08, 8, cor_centro)
    rot = matrix_rotation_z(22.5)
    partes.append((Objeto3D(v, i), rot))
    
    return partes

print("Função create_estrela_sol definida!")

Função create_estrela_sol definida!


### 8.4 Planeta (3D)
Composto por: corpo (esfera), faixa decorativa (torus), anéis externos (torus)

In [11]:
def create_planeta():
    """
    Cria um planeta com anéis (tipo Saturno).
    Retorna lista de objetos e suas transformações locais.
    """
    partes = []
    
    # Cores
    cor_planeta = [0.6, 0.4, 0.3]       # Marrom/laranja
    cor_faixa1 = [0.7, 0.5, 0.35]       # Faixa mais clara
    cor_anel = [0.8, 0.75, 0.6]         # Bege/dourado
    
    # Corpo do planeta (esfera principal)
    v, i = create_sphere(0.18, 24, 24, cor_planeta)
    partes.append((Objeto3D(v, i), matrix_identity()))
    
    # Faixa decorativa (torus fino)
    v, i = create_torus(0.18, 0.01, 24, 8, cor_faixa1)
    partes.append((Objeto3D(v, i), matrix_identity()))
    
    # Anel externo (torus)
    v, i = create_torus(0.3, 0.02, 32, 8, cor_anel)
    rot = matrix_rotation_x(75)
    partes.append((Objeto3D(v, i), rot))
    
    # Anel interno
    v, i = create_torus(0.25, 0.015, 32, 8, cor_anel)
    rot = matrix_rotation_x(75)
    partes.append((Objeto3D(v, i), rot))
    
    return partes

print("Função create_planeta definida!")

Função create_planeta definida!


### 8.5 Satélite (2D)
Composto por: corpo (cilindro), painéis solares (retângulos), antena (cone), prato (cone)

In [12]:
def create_satelite():
    """
    Cria um satélite composto por retângulos e painéis solares.
    Retorna lista de objetos e suas transformações locais.
    """
    partes = []
    
    # Cores
    cor_corpo = [0.7, 0.7, 0.75]        # Prata
    cor_painel = [0.2, 0.3, 0.6]        # Azul escuro (painéis solares)
    cor_antena = [0.9, 0.9, 0.9]        # Branco
    cor_detalhe = [1.0, 0.8, 0.2]       # Dourado
    
    # Corpo principal (retângulo/cilindro)
    v, i = create_cylinder(0.04, 0.1, 12, cor_corpo)
    partes.append((Objeto3D(v, i), matrix_identity()))
    
    # Painel solar esquerdo
    v, i = create_rectangle(0.15, 0.06, cor_painel)
    trans = matrix_translation(-0.11, 0, 0)
    partes.append((Objeto3D(v, i), trans))
    
    # Painel solar direito
    v, i = create_rectangle(0.15, 0.06, cor_painel)
    trans = matrix_translation(0.11, 0, 0)
    partes.append((Objeto3D(v, i), trans))
    
    # Estrutura do painel esquerdo
    v, i = create_rectangle(0.16, 0.005, cor_detalhe)
    trans = matrix_translation(-0.11, 0.03, 0.001)
    partes.append((Objeto3D(v, i), trans))
    
    # Estrutura do painel direito
    v, i = create_rectangle(0.16, 0.005, cor_detalhe)
    trans = matrix_translation(0.11, 0.03, 0.001)
    partes.append((Objeto3D(v, i), trans))
    
    # Antena (cone pequeno)
    v, i = create_cone(0.015, 0.06, 10, cor_antena)
    trans = matrix_translation(0, 0.08, 0)
    partes.append((Objeto3D(v, i), trans))
    
    # Prato da antena (cone achatado)
    v, i = create_cone(0.025, 0.02, 12, cor_detalhe)
    rot = matrix_rotation_x(180)
    trans = matrix_translation(0, 0.06, 0)
    partes.append((Objeto3D(v, i), matrix_multiply(trans, rot)))
    
    return partes

print("Função create_satelite definida!")

Função create_satelite definida!


### 8.6 Estrelas de Fundo

In [13]:
def create_estrelas_fundo():
    """
    Cria várias estrelas pequenas para o fundo.
    Retorna lista de objetos e suas posições.
    """
    import random
    random.seed(42)  # Para reprodutibilidade
    
    estrelas = []
    
    for _ in range(50):
        # Posição aleatória
        x = random.uniform(-1.8, 1.8)
        y = random.uniform(-1.0, 1.0)
        z = -0.5  # Fundo
        
        # Tamanho aleatório
        tamanho = random.uniform(0.005, 0.015)
        
        # Cor (variações de branco/amarelo)
        brilho = random.uniform(0.7, 1.0)
        cor = [brilho, brilho, random.uniform(0.8, 1.0) * brilho]
        
        # Cria estrela simples (esfera pequena)
        v, i = create_sphere(tamanho, 6, 6, cor)
        obj = Objeto3D(v, i)
        pos = matrix_translation(x, y, z)
        
        estrelas.append((obj, pos))
    
    return estrelas

print("Função create_estrelas_fundo definida!")

Função create_estrelas_fundo definida!


## 9. Callback de Teclado e Loop Principal

In [14]:
def key_callback(window, key, scancode, action, mods):
    """Callback para eventos de teclado."""
    global wireframe_mode, astronauta_pos, foguete_scale, estrela_rotation
    
    if action == glfw.PRESS or action == glfw.REPEAT:
        # Translação do astronauta (WASD)
        if key == glfw.KEY_W:
            astronauta_pos[1] += TRANSLATION_SPEED
        elif key == glfw.KEY_S:
            astronauta_pos[1] -= TRANSLATION_SPEED
        elif key == glfw.KEY_A:
            astronauta_pos[0] -= TRANSLATION_SPEED
        elif key == glfw.KEY_D:
            astronauta_pos[0] += TRANSLATION_SPEED
        
        # Escala do foguete (Z/X)
        elif key == glfw.KEY_Z:
            foguete_scale = max(0.3, foguete_scale - SCALE_SPEED)
        elif key == glfw.KEY_X:
            foguete_scale = min(2.0, foguete_scale + SCALE_SPEED)
        
        # Rotação da estrela (Q/E)
        elif key == glfw.KEY_Q:
            estrela_rotation += ROTATION_SPEED
        elif key == glfw.KEY_E:
            estrela_rotation -= ROTATION_SPEED
        
        # Toggle wireframe (P)
        elif key == glfw.KEY_P:
            wireframe_mode = not wireframe_mode
        
        # Sair (ESC)
        elif key == glfw.KEY_ESCAPE:
            glfw.set_window_should_close(window, True)

print("Callback de teclado definido!")

Callback de teclado definido!


## 10. Função Principal - Execução do Programa

In [ ]:
def main():
    """Função principal do programa."""
    global planeta_rotation, wireframe_mode
    
    # Inicializa GLFW
    if not glfw.init():
        raise RuntimeError("Falha ao inicializar GLFW")
    
    # Configurações da janela
    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
    glfw.window_hint(glfw.SAMPLES, 4)  # Anti-aliasing
    
    # Cria janela
    window = glfw.create_window(
        WINDOW_WIDTH, WINDOW_HEIGHT,
        "Computação Gráfica - Projeto 1: Astronauta no Espaço",
        None, None
    )
    
    if not window:
        glfw.terminate()
        raise RuntimeError("Falha ao criar janela GLFW")
    
    glfw.make_context_current(window)
    glfw.set_key_callback(window, key_callback)
    
    # Configurações OpenGL
    glEnable(GL_DEPTH_TEST)
    glEnable(GL_MULTISAMPLE)
    glClearColor(0.02, 0.02, 0.08, 1.0)  # Fundo azul escuro (espaço)
    
    # Cria programa de shaders (sem uniforms de matrizes)
    shader_program = create_shader_program()
    
    # Matriz de projeção ortográfica (aplicada via CPU nos vértices)
    aspect = WINDOW_WIDTH / WINDOW_HEIGHT
    projection = matrix_orthographic(-aspect, aspect, -1.0, 1.0, -10.0, 10.0)
    
    # Inclinação global para reforçar a percepção de profundidade
    scene_tilt = matrix_multiply(
        matrix_rotation_y(-20),
        matrix_rotation_x(12)
    )
    
    # Matriz base da cena: projeção * inclinação (pré-calculada)
    scene_base = matrix_multiply(projection, scene_tilt)
    
    # Cria objetos da cena
    print("Criando objetos da cena...")
    astronauta_partes = create_astronauta()
    foguete_partes = create_foguete()
    estrela_partes = create_estrela_sol()
    planeta_partes = create_planeta()
    satelite_partes = create_satelite()
    estrelas_fundo = create_estrelas_fundo()
    
    print("\n" + "="*60)
    print("CONTROLES:")
    print("="*60)
    print("W/A/S/D  - Mover astronauta (translação)")
    print("Z/X      - Diminuir/Aumentar foguete (escala)")
    print("Q/E      - Rotacionar estrela/sol")
    print("P        - Mostrar/ocultar malha poligonal (wireframe)")
    print("ESC      - Sair")
    print("="*60 + "\n")
    
    # Loop principal
    while not glfw.window_should_close(window):
        # Processa eventos
        glfw.poll_events()
        
        # Atualiza rotação automática do planeta
        planeta_rotation += 0.3
        
        # Limpa buffers
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
        
        # Configura modo de renderização
        if wireframe_mode:
            glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
        else:
            glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
        
        # Usa o shader program
        glUseProgram(shader_program)
        
        # ============================================================
        # Desenha estrelas de fundo (estáticas)
        # ============================================================
        for obj, pos in estrelas_fundo:
            combined = matrix_multiply(scene_base, pos)
            obj.draw(combined)
        
        # ============================================================
        # Desenha ASTRONAUTA (translação com WASD)
        # ============================================================
        astronauta_base = matrix_translation(
            astronauta_pos[0] - 0.5,
            astronauta_pos[1],
            astronauta_pos[2]
        )
        for obj, local_transform in astronauta_partes:
            combined = matrix_multiply(
                scene_base,
                matrix_multiply(astronauta_base, local_transform)
            )
            obj.draw(combined)
        
        # ============================================================
        # Desenha FOGUETE (escala com Z/X)
        # ============================================================
        foguete_base = matrix_translation(0.7, -0.3, 0)
        foguete_scale_mat = matrix_scale(foguete_scale, foguete_scale, foguete_scale)
        foguete_rot = matrix_rotation_z(-15)  # Inclinado
        foguete_transform = matrix_multiply(foguete_base, matrix_multiply(foguete_rot, foguete_scale_mat))
        for obj, local_transform in foguete_partes:
            combined = matrix_multiply(
                scene_base,
                matrix_multiply(foguete_transform, local_transform)
            )
            obj.draw(combined)
        
        # ============================================================
        # Desenha ESTRELA/SOL (rotação com Q/E)
        # ============================================================
        estrela_base = matrix_translation(-0.7, 0.6, 0)
        estrela_rot = matrix_rotation_z(estrela_rotation)
        estrela_transform = matrix_multiply(estrela_base, estrela_rot)
        for obj, local_transform in estrela_partes:
            combined = matrix_multiply(
                scene_base,
                matrix_multiply(estrela_transform, local_transform)
            )
            obj.draw(combined)
        
        # ============================================================
        # Desenha PLANETA (rotação automática)
        # ============================================================
        planeta_base = matrix_translation(0.9, 0.5, -0.2)
        planeta_scale_mat = matrix_scale(0.7, 0.7, 0.7)
        planeta_rot = matrix_rotation_y(planeta_rotation)
        planeta_transform = matrix_multiply(planeta_base, matrix_multiply(planeta_rot, planeta_scale_mat))
        for obj, local_transform in planeta_partes:
            combined = matrix_multiply(
                scene_base,
                matrix_multiply(planeta_transform, local_transform)
            )
            obj.draw(combined)
        
        # ============================================================
        # Desenha SATÉLITE (estático)
        # ============================================================
        satelite_base = matrix_translation(-0.9, -0.5, 0)
        satelite_rot = matrix_rotation_z(25)
        satelite_transform = matrix_multiply(satelite_base, satelite_rot)
        for obj, local_transform in satelite_partes:
            combined = matrix_multiply(
                scene_base,
                matrix_multiply(satelite_transform, local_transform)
            )
            obj.draw(combined)
        
        # Troca buffers
        glfw.swap_buffers(window)
    
    # Limpeza
    print("Encerrando programa...")
    
    # Libera recursos dos objetos
    for obj, _ in astronauta_partes:
        obj.cleanup()
    for obj, _ in foguete_partes:
        obj.cleanup()
    for obj, _ in estrela_partes:
        obj.cleanup()
    for obj, _ in planeta_partes:
        obj.cleanup()
    for obj, _ in satelite_partes:
        obj.cleanup()
    for obj, _ in estrelas_fundo:
        obj.cleanup()
    
    glDeleteProgram(shader_program)
    glfw.terminate()
    print("Programa encerrado com sucesso!")

print("Função main definida!")

Função main definida!


## 11. Executar o Programa

Execute a célula abaixo para iniciar a aplicação gráfica:

In [16]:
# Executa o programa principal
if __name__ == "__main__":
    main()

Criando objetos da cena...

CONTROLES:
W/A/S/D  - Mover astronauta (translação)
Z/X      - Diminuir/Aumentar foguete (escala)
Q/E      - Rotacionar estrela/sol
P        - Mostrar/ocultar malha poligonal (wireframe)
ESC      - Sair

Encerrando programa...
Programa encerrado com sucesso!


---

## Resumo dos Requisitos Atendidos

| Requisito | Implementação |
|-----------|---------------|
| 5+ objetos diferentes | Astronauta, Foguete, Estrela/Sol, Planeta, Satélite (+ estrelas de fundo) |
| 2+ objetos 3D | Astronauta (esferas+cilindros), Foguete (cone+cilindros), Planeta (esfera+torus) |
| Objetos compostos | Todos são composições de primitivas simples criadas manualmente |
| Sem projection/model/view no shader | Transformações aplicadas nos vértices via CPU antes do envio ao GPU |
| Translação | Astronauta (teclas WASD) |
| Escala | Foguete (teclas Z/X) |
| Rotação | Estrela/Sol (teclas Q/E); Planeta (automática) |
| Visualizar malha (wireframe) | Tecla P |
| Sem texturas | Cores definidas diretamente nos vértices |
| Sem iluminação | Cores fixas sem cálculos de luz |
| Sem câmera | Sem movimentação de câmera |
| Pipeline moderno OpenGL | Shaders GLSL 3.3, VAO/VBO/EBO |
| Sem funções deprecated | Nenhuma função obsoleta utilizada |